# Chapter 5 — Decoders in Action

Companion code for **Chapter 5** of *Build an Advanced RAG Application (From Scratch)*.

We turn from retrieval (Chapter 4) to generation. The decoder is the **producer** of the answer — and how we *prompt* it determines the quality of what comes out.

This notebook walks through four prompting styles:

1. **Basic** — single-line ask
2. **Structured** — explicit sections and constraints
3. **Few-shot** — show the model the pattern by example
4. **Chain-of-Thought (CoT)** — make the model reason step by step

By the end, we apply CoT to **analyze hotel search results** — which is exactly the seam Chapter 6 plugs retrieval into.

> Reusable code lives in `llm_client.py` and `prompts.py` next to this notebook.

## 0. Setup

This notebook calls the OpenAI API. Copy `.env.example` at the repo root to `.env` and add your `OPENAI_API_KEY` before running.

In [ ]:
import sys, os
sys.path.insert(0, '.')

from IPython.display import Markdown, display
from llm_client import generate_text
from prompts import (
    BASIC_HOTEL,
    STRUCTURED_HOTEL,
    few_shot_hotel,
    chain_of_thought_hotel,
    analyze_hotel_search_results,
)

## 1. Basic vs. structured prompting

The same model, the same topic, two prompts. The difference in answer quality is entirely on us.

In [ ]:
display(Markdown(generate_text(BASIC_HOTEL)))

In [ ]:
display(Markdown(generate_text(STRUCTURED_HOTEL)))

### Try it on a different topic

The pattern transfers — be specific about *what you want, in what shape*.

In [ ]:
basic = "Tell me about climate change"
engineered = '''Provide a comprehensive analysis of climate change, focusing on:
1. Current scientific consensus
2. Major environmental impacts
3. Mitigation strategies
Format this as a structured report with clear headings and evidence-based conclusions.'''

display(Markdown(generate_text(basic)))
print('---')
display(Markdown(generate_text(engineered)))

## 2. Few-shot prompting

When you can't fully specify the format in words, *show* it. Two or three labeled examples are usually enough.

In [ ]:
prompt = few_shot_hotel("Tell me about Paris hotels")
print(prompt)
print("\n--- Response ---\n")
display(Markdown(generate_text(prompt, temperature=0.7, max_tokens=200)))

## 3. Chain-of-Thought prompting

CoT asks the model to *show its work* — reasoning step by step before giving the final answer. It tends to:

- Improve correctness on multi-step problems
- Make wrong answers easier to spot (the bad reasoning is visible)
- Cost more tokens (it produces more text)


In [ ]:
prompt = chain_of_thought_hotel("Tell me about Paris hotels")
display(Markdown(generate_text(prompt, temperature=0.7, max_tokens=400)))

## 4. CoT applied to retrieval results — the bridge into Chapter 6

This is the prompt pattern Chapter 6 uses to summarize FAISS / Qdrant search results into a grounded answer.

For demonstration we paste in some example retrieval output. In Chapter 6 the `results` string comes from the search pipeline programmatically.

In [ ]:
query = "Hotel near the Louvre with great food nearby."
results = '''
Top hotel with similar reviews using FAISS:
1. Grand Hotel du Palais Royal
   Review: Great hotel located in the best part of town just next to the Louvre.
           Walking distance to the Louvre, Notre Dame, and shops. Very clean and
           the service was wonderful.
   Distance: 0.7877

2. Hotel Malte - Astotel
   Review: Excellent hotel all round. Ideal location for the Louvre, city centre
           shops, and plenty of metro stations within a few minutes walk. There
           are numerous restaurants and cafes close by.
   Distance: 0.7820

3. Hotel du Continent
   Review: Lovely small hotel in a fantastic location, just a few minutes walk
           from the Louvre. The breakfast is great with plenty of choice.
   Distance: 0.7654
'''.strip()

prompt = analyze_hotel_search_results(query, results)
display(Markdown(generate_text(prompt, model="gpt-4o", max_tokens=800)))

## What's next

In Chapter 6 we replace the hard-coded `results` block with a live FAISS / Qdrant search and stream the LLM's answer back to the user — a complete RAG loop.